# 01 — Data Exploration

Load g.tec recoveriX .mat files, inspect signal quality, visualize raw EEG,
compute PSDs, and assess epoch quality per patient.

**Prerequisites:** Place `.mat` files in `../data/`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import mne

from src.loading import load_all_patients, CH_NAMES, LEFT_IDX, RIGHT_IDX
from src.lateralization import compute_laterality_index, laterality_report

%matplotlib inline
mne.set_log_level('WARNING')

## 1.1 Load All Patients

In [ ]:
patient_data = load_all_patients('../data/')

for pid, (X, y, epochs) in patient_data.items():
    n_left = (y == 0).sum()
    n_right = (y == 1).sum()
    print(f'{pid}: {X.shape[0]} epochs ({n_left} left, {n_right} right), '
          f'{X.shape[1]} channels, {X.shape[2]} samples')

## 1.2 Raw Signal Visualization

Plot a few seconds of raw EEG for the first patient to check signal quality.

In [ ]:
first_pid = sorted(patient_data.keys())[0]
X0, y0, epochs0 = patient_data[first_pid]

# Plot average epoch per class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cls, label in enumerate(['Left MI', 'Right MI']):
    mask = y0 == cls
    avg = X0[mask].mean(axis=0)  # (16, n_times)
    times = np.arange(avg.shape[1]) / epochs0.info['sfreq']
    for ch in range(avg.shape[0]):
        axes[cls].plot(times, avg[ch] * 1e6, alpha=0.5, linewidth=0.8)
    axes[cls].set_title(f'{label} — Grand Average ({first_pid})')
    axes[cls].set_xlabel('Time (s)')
    axes[cls].set_ylabel('Amplitude (µV)')
plt.tight_layout()
plt.show()

## 1.3 Power Spectral Density

In [ ]:
epochs0.compute_psd(fmin=1, fmax=50, picks='eeg').plot(show=True)

## 1.4 Topographic Maps (Mu Band ERD)

In [ ]:
from src.visualization import plot_topographic_erd

for pid, (X, y, epochs) in patient_data.items():
    print(f'\n--- {pid} ---')
    plot_topographic_erd(epochs, epochs.info['sfreq'], CH_NAMES, band=(8, 13))

## 1.5 Lateralization Analysis

In [ ]:
report = laterality_report(patient_data)
print(report)

In [ ]:
from src.visualization import plot_laterality_comparison

li_results = {}
for pid, (X, y, epochs) in patient_data.items():
    li_results[pid] = compute_laterality_index(X, epochs.info['sfreq'])

plot_laterality_comparison(li_results, save_path='../results/laterality_comparison.png')